# Lab — Assistant Support Client avec un LLM

**Objectif du projet :** construire, étape par étape, un assistant qui :
1. **répond aux réclamations** clients avec un **ton défini** (professionnel, empathique) ;
2. **refuse poliment** les demandes **hors périmètre** (hors du sujet support) ;
3. produit un **résumé structuré** de chaque échange (au format JSON).

>  **Modèle utilisé :** on n'utilise PAS Claude ici. On passe par les **Inference Providers de Hugging Face**
> en choisissant **Groq** comme fournisseur. Groq fait tourner des **modèles ouverts** (Llama, gpt-oss, Qwen...)
> avec une inférence très rapide (technologie LPU). Il te faut juste un **token Hugging Face** (gratuit).


## Étape 1 — Installation et connexion (Hugging Face + Groq)

> Remplace `"hf_xxxxx"` par ton **token Hugging Face**.
> Tu l'obtiens gratuitement sur : https://huggingface.co/settings/tokens (un token en lecture suffit).
> Ne partage jamais ton token publiquement.

On installe la librairie `huggingface_hub`, puis on crée un client en précisant **`provider="groq"`**.
Le même token HF sert à router les requêtes vers Groq (facturation aux tarifs du fournisseur ;
les comptes gratuits ont un petit quota).


In [ ]:
# À EXÉCUTER — installation (sur Google Colab, décommente la ligne suivante)
# !pip install huggingface_hub

from huggingface_hub import InferenceClient
import json

# Renseigne ton token Hugging Face ici (commence par "hf_")
HF_TOKEN = "This is private"

client = InferenceClient(
    provider="groq",       # ← on demande explicitement le fournisseur Groq
    api_key=HF_TOKEN,
)

# Un modèle ouvert servi par Groq. Alternatives possibles :
#   "openai/gpt-oss-120b"
#   "meta-llama/Llama-3.3-70B-Instruct"
#   "Qwen/Qwen3-32B"
MODEL = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

print("Client Groq (via Hugging Face) prêt")

Client Groq (via Hugging Face) prêt


## Étape 2 — Une fonction utilitaire pour appeler le modèle

On crée une petite fonction `demander()` pour ne pas réécrire l'appel à chaque fois.
Elle prend le **system prompt** et le **message du client**, construit la liste `messages`
(avec le rôle `"system"` en tête), puis renvoie le **texte** de la réponse.


In [6]:
# À EXÉCUTER
def demander(system_prompt, message_client, temperature=0.3, max_tokens=600):
    """Envoie un message au modèle (Groq via HF) et renvoie le texte de sa réponse."""
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},   # ← les règles / le ton
            {"role": "user",   "content": message_client},   # ← le message du client
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    # Style OpenAI : on récupère le texte dans choices[0].message.content
    return completion.choices[0].message.content

print("Fonction 'demander' définie")

Fonction 'demander' définie


## Étape 3 — Définir le TON (première version du system prompt)

On commence simple : un assistant qui répond avec un **ton défini**.
Décris précisément le **rôle**, le **ton**, et le **public**.

Conseil : plus les instructions sont concrètes, meilleur est le résultat.
Évite *"sois gentil"* → préfère *"reconnais la frustration du client dès la première phrase"*.


In [7]:
# À EXÉCUTER — version 1 : le ton seulement
SYSTEM_V1 = """
Tu es un assistant service client pour une boutique en ligne.
Ton rôle est de répondre aux réclamations des clients avec un ton professionnel et empathique.
Dès la première phrase, reconnais la frustration ou l'insatisfaction du client.
Utilise un langage clair, poli et rassurant.
Propose toujours une solution concrète ou une prochaine étape.
Réponds en français.
"""

reclamation = "Bonjour, j'ai commandé un casque il y a 10 jours et je n'ai toujours rien reçu. C'est inadmissible !"

print(demander(SYSTEM_V1, reclamation))

Bonjour, je comprends parfaitement votre frustration et votre déception face à la situation. Je suis désolé que vous n'ayez pas reçu votre casque dans les délais attendus. Nous nous excusons pour la gêne occasionnée et je suis là pour vous aider à résoudre ce problème.

Pouvez-vous me confirmer votre numéro de commande afin que je puisse vérifier l'état de votre livraison ? Cela me permettra de voir si votre commande a bien été expédiée et si elle a rencontré des problèmes en cours de route.

Je vous propose de suivre l'état de votre commande et de vous fournir une mise à jour sur la date de livraison prévue. Si nécessaire, nous pourrons également envisager une réexpédition ou un remboursement, selon vos préférences. Comment puis-je procéder pour vous aider ?


**Observe :** le ton est-il empathique ? La réponse reconnaît-elle le problème ?
Rejoue la cellule 2-3 fois : à `temperature=0.3`, les réponses restent stables et professionnelles.


## Étape 4 — Ajouter le REFUS POLI (hors périmètre)

Un bon assistant support **reste dans son rôle**. S'il reçoit une question hors sujet
(ex : *"donne-moi une recette de couscous"* ou *"aide-moi à coder"*), il doit **refuser poliment**
et recentrer.

On ajoute donc une section **PÉRIMÈTRE** dans le system prompt.


In [8]:
# À EXÉCUTER — version 2 : ton + refus hors périmètre
SYSTEM_V2 = """
Tu es un assistant service client pour une boutique en ligne.
Ton rôle est de répondre aux réclamations des clients avec un ton professionnel et empathique.
Dès la première phrase, reconnais la frustration ou l'insatisfaction du client.
Utilise un langage clair, poli et rassurant.
Propose toujours une solution concrète ou une prochaine étape.
Réponds en français.

PÉRIMÈTRE STRICT :
Tu traites UNIQUEMENT les demandes liées au service client : commandes, livraisons, remboursements, retours, produits défectueux et réclamations.
Si le client pose une question hors de ce périmètre (recette de cuisine, aide à la programmation, jeux, etc.), refuse poliment et recentre la conversation sur le support client.
Ne réponds jamais au contenu hors périmètre, même partiellement.
"""

# Test 1 : une vraie réclamation
print("=== CAS 1 : réclamation valide ===")
print(demander(SYSTEM_V2, "Mon colis est arrivé cassé, je veux un remboursement."))

print("\n=== CAS 2 : hors périmètre ===")
print(demander(SYSTEM_V2, "Peux-tu me donner une recette de tajine ?"))

=== CAS 1 : réclamation valide ===
Je suis désolé d'apprendre que votre colis est arrivé en mauvais état. Je comprends parfaitement votre frustration et je suis là pour vous aider à résoudre ce problème le plus rapidement possible.

Pouvez-vous me donner plus de détails sur votre commande, comme le numéro de commande ou la référence du produit ? Cela me permettra de localiser votre dossier et de traiter votre demande de remboursement de manière efficace.

En attendant, je vous assure que nous allons prendre les mesures nécessaires pour régler cette situation. Nous proposons généralement un remboursement ou un échange pour les produits endommagés. Je vous tiendrai informé des prochaines étapes.

=== CAS 2 : hors périmètre ===
Je comprends que vous cherchez une délicieuse recette de tajine, mais je suis désolé de vous informer que mon rôle est de traiter les demandes liées au service client de notre boutique en ligne, notamment les commandes, les livraisons, les remboursements, les retou

**Observe :** au CAS 2, l'assistant doit **refuser poliment** et **ne pas** donner la recette.
Si jamais il répond quand même à la recette, c'est que ton instruction de périmètre n'est pas assez ferme :
c'est exactement le genre de bug de prompt à corriger. 


## Étape 5 — Produire un RÉSUMÉ STRUCTURÉ (JSON)

Pour être exploitable par un outil (CRM, ticketing...), on veut aussi un **résumé structuré**.
On demande au modèle de renvoyer un **objet JSON** avec des champs précis.


In [9]:
# À EXÉCUTER — un system prompt dédié au résumé structuré
SYSTEM_RESUME = """
Tu es un outil d'analyse de réclamations clients.
Pour chaque message reçu, tu dois produire UNIQUEMENT un objet JSON valide, sans aucun texte avant ou après.
Le JSON doit contenir exactement ces champs :
{
  "categorie": "<type de réclamation : livraison / remboursement / produit_defectueux / autre>",
  "urgence": "<niveau d'urgence : faible / moyenne / haute>",
  "sentiment": "<sentiment du client : neutre / insatisfait / tres_insatisfait / furieux>",
  "resume": "<résumé de la réclamation en une phrase>",
  "action_recommandee": "<action à entreprendre par le service client>"
}
UNIQUEMENT le JSON, aucun autre texte, aucun bloc de code markdown.
"""

message = "Ça fait 3 semaines que j'attends mon remboursement, personne ne répond, je suis furieux."

sortie = demander(SYSTEM_RESUME, message, temperature=0)
print(sortie)

{"categorie": "remboursement", "urgence": "haute", "sentiment": "furieux", "resume": "Le client attend son remboursement depuis 3 semaines sans réponse.", "action_recommandee": "Contacter le client pour expliquer les raisons du retard et fournir une date de remboursement."}


In [10]:
# À EXÉCUTER — transformer le texte JSON en vrai dictionnaire Python
try:
    donnees = json.loads(sortie)
    print("JSON valide ")
    print("Catégorie :", donnees["categorie"])
    print("Urgence   :", donnees["urgence"])
    print("Action    :", donnees["action_recommandee"])
except json.JSONDecodeError:
    print("Le modèle n'a pas renvoyé du JSON pur. Renforce l'instruction de format.")

JSON valide 
Catégorie : remboursement
Urgence   : haute
Action    : Contacter le client pour expliquer les raisons du retard et fournir une date de remboursement.


**Astuce pro :** on met `temperature=0` pour les sorties structurées → plus fiable et reproductible.
Si `json.loads` échoue, c'est presque toujours parce que le modèle a ajouté du texte autour du JSON :
insiste dans le prompt (*"UNIQUEMENT le JSON, aucun autre texte"*).


## Étape 6 — Assembler l'assistant complet

On combine tout dans une seule fonction `assistant_support()` qui, pour un message client, renvoie :
- **la réponse** au client (ton + refus si besoin) ;
- **le résumé structuré** (JSON).

C'est le livrable principal du projet.


In [11]:
# À EXÉCUTER
def assistant_support(message_client):
    """Traite une réclamation : renvoie (reponse_client, resume_json)."""

    # 1) La réponse au client (ton + périmètre)
    reponse_client = demander(SYSTEM_V2, message_client)

    # 2) Le résumé structuré (format machine)
    resume_txt = demander(SYSTEM_RESUME, message_client, temperature=0)
    try:
        resume = json.loads(resume_txt)
    except json.JSONDecodeError:
        resume = {"erreur": "JSON non valide", "brut": resume_txt}

    return reponse_client, resume


# Test complet
msg = "Bonjour, le téléphone que j'ai acheté ne s'allume plus après 2 jours. Je veux un échange !"
reponse, resume = assistant_support(msg)

print("RÉPONSE AU CLIENT :\n")
print(reponse)
print("\n" + "="*50)
print("RÉSUMÉ STRUCTURÉ :\n")
print(json.dumps(resume, indent=2, ensure_ascii=False))

RÉPONSE AU CLIENT :

Je suis désolé d'apprendre que votre téléphone ne fonctionne plus après seulement 2 jours. Je comprends parfaitement votre frustration et je suis là pour vous aider à résoudre ce problème.

Tout d'abord, je vous assure que nous prenons très au sérieux les problèmes de qualité de nos produits. Pouvez-vous me donner plus de détails sur votre achat, tels que la référence du téléphone, la date d'achat et le numéro de commande ? Cela me permettra de vérifier les informations et de vous proposer une solution adaptée.

En attendant, je vous propose de procéder à un échange ou à un remboursement, selon vos préférences. Nous avons une garantie de 2 ans sur nos téléphones, donc vous êtes encore dans le délai de garantie.

Quelle est votre préférence ? Souhaitez-vous un échange contre un nouveau téléphone ou un remboursement ? Je suis à votre écoute pour vous aider à résoudre ce problème le plus rapidement possible.

RÉSUMÉ STRUCTURÉ :

{
  "categorie": "produit_defectueux",
